# 05 · Cross-Sectional Evaluation
Run the full pipeline on a basket of large-cap US stocks and compare the LSTM strategy against buy & hold on the held-out test period.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from quant_dl.data import download
from quant_dl.pipeline import run_experiment

In [2]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'JPM', 'XOM']
START, END = '2018-01-01', '2025-01-01'

In [3]:
rows = []
cost_tables = {}
for t in TICKERS:
    print(f'--- {t} ---', flush=True)
    df = download(t, START, END)
    r = run_experiment(df, seed=42)
    rows.append({
        'ticker': t,
        'strat_total_return': r['strategy']['total_return'],
        'strat_annualized': r['strategy']['annualized_return'],
        'strat_sharpe': r['strategy']['sharpe_ratio'],
        'strat_max_dd': r['strategy']['max_drawdown'],
        'strat_win_rate': r['strategy']['win_rate'],
        'n_trades': r['strategy']['n_trades'],
        'bh_total_return': r['buy_and_hold']['total_return'],
        'bh_sharpe': r['buy_and_hold']['sharpe_ratio'],
        'bh_max_dd': r['buy_and_hold']['max_drawdown'],
    })
    cost_tables[t] = r['cost_sensitivity']
summary = pd.DataFrame(rows).set_index('ticker')
summary

--- AAPL ---


--- MSFT ---


--- GOOGL ---


--- AMZN ---


--- NVDA ---


--- META ---


--- JPM ---


--- XOM ---


,strat_total_return,strat_annualized,strat_sharpe,strat_max_dd,strat_win_rate,n_trades,bh_total_return,bh_sharpe,bh_max_dd
ticker,,,,,,,,,
AAPL,0.038244,0.054314,0.382737,-0.170010,0.461538,13,0.290332,1.467033,-0.153548
MSFT,0.064080,0.091475,0.595669,-0.097344,0.619048,21,0.144643,0.921890,-0.154868
GOOGL,0.237686,0.350557,1.439611,-0.129137,0.666667,12,0.371956,1.502011,-0.221376
AMZN,0.463172,0.709800,1.805073,-0.192130,0.727273,11,0.440776,1.708415,-0.194900
NVDA,2.344190,4.481073,3.475919,-0.221971,0.800000,10,1.789256,2.630045,-0.270468
META,0.036020,0.051133,0.317758,-0.231122,0.600000,10,0.681057,1.898985,-0.184264
JPM,0.339907,0.510365,1.715101,-0.101305,0.833333,12,0.472164,2.091831,-0.101305
XOM,0.070387,0.100603,0.792062,-0.107669,0.555556,9,0.097346,0.685328,-0.151496


## Summary
Formatted for the README results table.

In [4]:
pct_cols = ['strat_total_return', 'strat_annualized', 'strat_max_dd', 'strat_win_rate',
            'bh_total_return', 'bh_max_dd']
fmt = summary.copy()
for c in pct_cols:
    fmt[c] = (fmt[c] * 100).round(1).astype(str) + '%'
for c in ['strat_sharpe', 'bh_sharpe']:
    fmt[c] = fmt[c].round(2)
fmt['n_trades'] = fmt['n_trades'].astype(int)
fmt.to_markdown()

'| ticker   | strat_total_return   | strat_annualized   |   strat_sharpe | strat_max_dd   | strat_win_rate   |   n_trades | bh_total_return   |   bh_sharpe | bh_max_dd   |\n|:---------|:---------------------|:-------------------|---------------:|:---------------|:-----------------|-----------:|:------------------|------------:|:------------|\n| AAPL     | 3.8%                 | 5.4%               |           0.38 | -17.0%         | 46.2%            |         13 | 29.0%             |        1.47 | -15.4%      |\n| MSFT     | 6.4%                 | 9.1%               |           0.6  | -9.7%          | 61.9%            |         21 | 14.5%             |        0.92 | -15.5%      |\n| GOOGL    | 23.8%                | 35.1%              |           1.44 | -12.9%         | 66.7%            |         12 | 37.2%             |        1.5  | -22.1%      |\n| AMZN     | 46.3%                | 71.0%              |           1.81 | -19.2%         | 72.7%            |         11 | 44.1%           

In [5]:
print(fmt.to_markdown())

| ticker   | strat_total_return   | strat_annualized   |   strat_sharpe | strat_max_dd   | strat_win_rate   |   n_trades | bh_total_return   |   bh_sharpe | bh_max_dd   |
|:---------|:---------------------|:-------------------|---------------:|:---------------|:-----------------|-----------:|:------------------|------------:|:------------|
| AAPL     | 3.8%                 | 5.4%               |           0.38 | -17.0%         | 46.2%            |         13 | 29.0%             |        1.47 | -15.4%      |
| MSFT     | 6.4%                 | 9.1%               |           0.6  | -9.7%          | 61.9%            |         21 | 14.5%             |        0.92 | -15.5%      |
| GOOGL    | 23.8%                | 35.1%              |           1.44 | -12.9%         | 66.7%            |         12 | 37.2%             |        1.5  | -22.1%      |
| AMZN     | 46.3%                | 71.0%              |           1.81 | -19.2%         | 72.7%            |         11 | 44.1%             |   

## Aggregate
Median across tickers is more robust than the mean to outliers.

In [6]:
summary.median()

strat_total_return     0.154037
strat_annualized       0.225580
strat_sharpe           1.115836
strat_max_dd          -0.149574
strat_win_rate         0.642857
n_trades              11.500000
bh_total_return        0.406366
bh_sharpe              1.605213
bh_max_dd             -0.169566
dtype: float64

## Transaction-cost sensitivity (example: AAPL)
How the strategy's edge decays as fees rise.

In [7]:
cost_tables['AAPL']

,fee,total_return,sharpe_ratio,n_trades
0,0.0000,0.065592,0.585814,13
1,0.0005,0.051829,0.484323,13
2,0.0010,0.038244,0.382737,13
3,0.0020,0.011597,0.179589,13
